# Ihminen-silmukassa-työnkulku Microsoft Agent Frameworkilla

## 🎯 Oppimistavoitteet

Tässä muistikirjassa opit toteuttamaan **ihminen-silmukassa**-työnkulkuja Microsoft Agent Frameworkin `RequestInfoExecutor`-luokan avulla. Tämä tehokas malli mahdollistaa tekoälyn työnkulkujen pysäyttämisen ihmisen syötteen keräämiseksi, tehden agenteistasi interaktiivisia ja antaen ihmisille hallinnan kriittisissä päätöksissä.

## 🔄 Mitä tarkoittaa Ihminen-silmukassa?

**Ihminen-silmukassa (HITL)** on suunnittelumalli, jossa tekoälyagentit pysäyttävät suorituksen pyytääkseen ihmisen syötettä ennen jatkamista. Tämä on välttämätöntä:

- ✅ **Kriittiset päätökset** - Saat ihmisen hyväksynnän ennen tärkeiden toimien suorittamista
- ✅ **Epätietoiset tilanteet** - Anna ihmisten selventää, kun tekoäly on epävarma
- ✅ **Käyttäjien mieltymykset** - Kysy käyttäjiltä valinta useiden vaihtoehtojen välillä
- ✅ **Sääntöjen ja turvallisuuden noudattaminen** - Varmista ihmisen valvonta säännellyissä toiminnoissa
- ✅ **Interaktiiviset kokemukset** - Rakenna keskustelevia agenteja, jotka reagoivat käyttäjän syötteisiin

## 🏗️ Miten se toimii Microsoft Agent Frameworkissa

Kehys tarjoaa kolme keskeistä komponenttia HITL:lle:

1. **`RequestInfoExecutor`** - Erityinen suoritin, joka pysäyttää työnkulun ja lähettää `RequestInfoEvent`-tapahtuman
2. **`RequestInfoMessage`** - Perusluokka tyypitetyille pyyntökuormille, jotka lähetetään ihmisille
3. **`RequestResponse`** - Yhdistää ihmisten vastaukset alkuperäisiin pyyntöihin käyttäen `request_id`:tä

**Työnkulku-malli:**
```
Agent detects need for input
    ↓
Sends message to RequestInfoExecutor
    ↓
Workflow pauses & emits RequestInfoEvent
    ↓
Application collects human input (console, UI, etc.)
    ↓
Application sends RequestResponse via send_responses_streaming()
    ↓
Workflow resumes with human input
```

## 🏨 Esimerkki: Hotellivarauksen tekeminen käyttäjän vahvistuksella

Rakennamme ehdollisen työnkulun päälle lisäämällä ihmisen vahvistuksen **ennen** vaihtoehtoisten kohteiden ehdottamista:

1. Käyttäjä pyytää matkakohdetta (esim. "Pariisi")
2. `availability_agent` tarkistaa ovatko huoneet saatavilla
3. **Jos huoneita ei ole** → `confirmation_agent` kysyy "Haluatko nähdä vaihtoehtoja?"
4. Työnkulku **pysähtyy** käyttäen `RequestInfoExecutor`-luokkaa
5. **Ihminen vastaa** "kyllä" tai "ei" konsolin syötteenä
6. `decision_manager` reitittää vastauksen perusteella:
   - **Kyllä** → Näytä vaihtoehtoiset matkakohteet
   - **Ei** → Peruuta varauspyyntö
7. Näytä lopputulos

Tämä näyttää, miten käyttäjille annetaan kontrolli agentin ehdotuksiin!

---

Aloitetaanpa! 🚀


## Vaihe 1: Tuo vaaditut kirjastot

Tuomme mukaan Agent Frameworkin peruskomponentit sekä **human-in-the-loop -erityiset luokat**:
- `RequestInfoExecutor` - Suorittaja, joka keskeyttää työnkulun ihmisen syötteen odottamiseksi
- `RequestInfoEvent` - Tapahtuma, joka lähetetään, kun ihmisen syöte pyydetään
- `RequestInfoMessage` - Pohjaluokka tyypitetyille pyyntöpainolasteille
- `RequestResponse` - Yhdistää ihmisen vastaukset pyyntöihin
- `WorkflowOutputEvent` - Tapahtuma työnkulun tulosten havaitsemiseksi


In [ ]:
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    Executor,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowRunState,          # Enum of workflow run states
    executor,
    handler,
    response_handler,          # NEW: decorator for the human-feedback response handler
    tool,
)

from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")
print("🔄 Human-in-the-loop uses ctx.request_info() + @response_handler")

✅ All imports successful!
🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse


## Vaihe 2: Määrittele Pydantic-mallit rakenteellisille vastauksille

Nämä mallit määrittelevät **kaavan**, jonka agentit palauttavat. Säilytämme kaikki ehdollisesta työnkulusta tutut mallit ja lisäämme:

**Uutta ihmisen osallistumisen ketjussa tueksi:**
- `HumanFeedbackRequest` - `RequestInfoMessage`-aliluokka, joka määrittelee ihmisille lähetettävän pyyntölatauksen
  - Sisältää `prompt` (kysymyksen) ja `destination` (konteksti poissaolevasta kaupungista)


In [ ]:
# Existing models from conditional workflow
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""
    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""
    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""
    destination: str
    action: str
    message: str


# NEW: Pydantic model for agent's response format
class ConfirmationQuestion(BaseModel):
    """
    Pydantic model used by confirmation_agent's response_format.
    This is what the agent will output as JSON.
    """
    question: str  # The question to ask the user
    destination: str  # The unavailable destination for context


# NEW: Plain dataclass payload for ctx.request_info()
@dataclass
class HumanFeedbackRequest:
    """
    Request sent to RequestInfoExecutor asking if user wants alternatives.
    
    MUST be a dataclass subclassing RequestInfoMessage for type compatibility.
    This is what gets sent to the RequestInfoExecutor.
    """
    prompt: str = ""  # The question to ask the user
    destination: str = ""  # The unavailable destination for context


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")
print("   - ConfirmationQuestion (agent response format) 🆕")
print("   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)
   - ConfirmationQuestion (agent response format) 🆕
   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕


## Vaihe 3: Luo hotellivarauksen työkalu

Sama työkalu kuin ehdollisessa työnkulussa – tarkistaa, onko kohteessa vapaita huoneita.


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

✅ hotel_booking tool created with @ai_function decorator


## Vaihe 4: Määritä ehdolliset funktiot reititykselle

Tarvitsemme **neljä ehdollista funktiota** ihmisen osallistamaa työnkulkua varten:

**Ehdollisesta työnkulusta:**
1. `has_availability_condition` - Reitittää, kun hotelleja ON saatavilla
2. `no_availability_condition` - Reitittää, kun hotelleja EI OLE saatavilla

**Uutta ihmisen osallistama työnkulku:**
3. `user_wants_alternatives_condition` - Reitittää, kun käyttäjä vastaa "kyllä" vaihtoehdoille
4. `user_declines_alternatives_condition` - Reitittää, kun käyttäjä vastaa "ei" vaihtoehdoille


In [24]:
# Existing condition functions from conditional workflow
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )
        return result.has_availability
    except Exception as e:
        display(HTML(f"""<div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'><strong>⚠️  Error:</strong> {str(e)}</div>"""))
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )
        return not result.has_availability
    except Exception as e:
        return False


# NEW: Condition functions for human-in-the-loop routing
def user_wants_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user WANTS to see alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            wants_alternatives = "wants to see alternative" in msg_text or "want to see alternative" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                    <strong>🔍 User Decision:</strong> User wants alternatives = <strong>{wants_alternatives}</strong>
                </div>
            """)
            )
            
            return wants_alternatives
    
    return False
def user_declines_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user DECLINES alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            declined = "declined" in msg_text or "has declined" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fce4ec; border-left: 4px solid #c2185b; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 User Decision:</strong> User declined alternatives = <strong>{declined}</strong>
                </div>
            """)
            )
            
            return declined
    
    return False
print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")
print("   - user_wants_alternatives_condition (routes when user says yes) 🆕")
print("   - user_declines_alternatives_condition (routes when user says no) 🆕")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)
   - user_wants_alternatives_condition (routes when user says yes) 🆕
   - user_declines_alternatives_condition (routes when user says no) 🆕


## Vaihe 5: Luo Decision Manager -suorittaja

Tämä on **ihmisen osallistumismallin ydin**! `DecisionManager` on mukautettu `Executor`, joka:

1. **Vastaanottaa ihmisen palautetta** `RequestResponse`-olioiden kautta
2. **Käsittelee käyttäjän päätöksen** (kyllä/ei)
3. **Ohjaa työnkulkua** lähettämällä viestejä sopiville agenteille

Keskeiset ominaisuudet:
- Käyttää `@handler`-koristetta altistaakseen metodit työnkulun vaiheiksi
- Vastaanottaa `RequestResponse[HumanFeedbackRequest, str]`, joka sisältää sekä alkuperäisen pyynnön että käyttäjän vastauksen
- Tuottaa yksinkertaisia "kyllä" tai "ei" -viestejä, jotka laukaisevat ehtofunktioitamme


In [ ]:
class DecisionManager(Executor):
    """
    Coordinates workflow routing based on human feedback.
    
    This executor receives RequestResponse objects from the RequestInfoExecutor
    and makes routing decisions by sending simple messages that trigger
    condition functions.
    """

    def __init__(self, id: str | None = None):
        super().__init__(id=id or "decision_manager")

    @handler
    async def on_confirmation(
        self,
        response: AgentExecutorResponse,
        ctx: WorkflowContext,
    ) -> None:
        """Parse the confirmation question and pause the workflow for human input."""
        confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                <strong>🔄 Requesting human input:</strong> {confirmation.question}
            </div>
        """)
        )
        # Pause the workflow; the human reply (a string) is delivered to on_human_feedback below.
        await ctx.request_info(
            request_data=HumanFeedbackRequest(
                prompt=confirmation.question,
                destination=confirmation.destination,
            ),
            response_type=str,
        )

    @response_handler
    async def on_human_feedback(
        self,
        original_request: HumanFeedbackRequest,
        feedback: str,
        ctx: WorkflowContext[AgentExecutorRequest, str],
    ) -> None:
        """Route the workflow based on the human's yes/no reply."""
        user_reply = (feedback or "").strip().lower()
        destination = original_request.destination or "unknown"

        display(
            HTML(f"""
            <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
                <strong>🎯 Decision Manager:</strong> Processing user reply: <strong>"{user_reply}"</strong> for {destination}
            </div>
        """)
        )

        if user_reply == "yes":
            display(
                HTML("""
                <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                    <strong>➡️  Routing:</strong> User wants alternatives → Will route to alternative_agent
                </div>
            """)
            )
            # Create and send a message for the alternative_agent
            user_msg = Message(
                role="user",
                contents=[f"The user wants to see alternative destinations near {destination}. Please suggest one."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        elif user_reply == "no":
            display(
                HTML("""
                <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 Routing:</strong> User declined alternatives → cancellation_agent
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        else:
            # Handle unexpected input - treat as decline
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                    <strong>⚠️  Warning:</strong> Unexpected input "{user_reply}" - treating as decline
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


print("✅ DecisionManager executor created (@handler pauses via request_info, @response_handler routes)")

✅ DecisionManager executor created with @handler method for human feedback


## Vaihe 6: Luo mukautettu näyttösuorittaja

Sama näyttösuorittaja kuin ehdollisessa työnkulussa - tuottaa lopulliset tulokset työnkulun tulosteena.


In [26]:
@executor(id="prepare_human_request")
async def prepare_human_request(
    response: AgentExecutorResponse, 
    ctx: WorkflowContext[HumanFeedbackRequest]
) -> None:
    """
    Transform agent response into HumanFeedbackRequest for RequestInfoExecutor.
    
    This executor bridges the type gap between:
    - confirmation_agent outputs AgentExecutorResponse with ConfirmationQuestion JSON
    - request_info_executor expects HumanFeedbackRequest (RequestInfoMessage dataclass)
    """
    display(
        HTML("""
        <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Transform:</strong> Converting ConfirmationQuestion to HumanFeedbackRequest
        </div>
    """)
    )
    
    # Parse the agent's Pydantic output (ConfirmationQuestion)
    confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
    
    # Convert to HumanFeedbackRequest dataclass for RequestInfoExecutor
    feedback_request = HumanFeedbackRequest(
        prompt=confirmation.question,
        destination=confirmation.destination
    )
    
    # Send the properly typed RequestInfoMessage to the RequestInfoExecutor
    await ctx.send_message(feedback_request)


@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ prepare_human_request executor created with @executor decorator")
print("✅ display_result executor created with @executor decorator")

✅ prepare_human_request executor created with @executor decorator
✅ display_result executor created with @executor decorator


## Vaihe 7: Lataa ympäristömuuttujat

Määritä LLM-asiakas (Microsoft Foundry / Azure OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

# Microsoft Foundry provider with keyless AzureCliCredential auth (run `az login`).
# Matches the pattern used across lessons 01-13 and the other Lesson 14 notebooks.
chat_client = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ Chat client configured with Microsoft Foundry")


✅ Chat client configured with GitHub Models


## Vaihe 8: Luo tekoälyagentit ja suorittajat

Luomme **kuusi työnkulun komponenttia**:

**Agentit (kääritty AgentExecutoriin):**
1. **availability_agent** - Tarkistaa hotellin saatavuuden työkalun avulla
2. **confirmation_agent** - 🆕 Valmistelee ihmisen vahvistuspyynnön
3. **alternative_agent** - Ehdottaa vaihtoehtoisia kaupunkeja (kun käyttäjä sanoo kyllä)
4. **booking_agent** - Kannustaa varaamaan (kun huoneita on saatavilla)
5. **cancellation_agent** - 🆕 Käsittelee peruutusviestin (kun käyttäjä sanoo ei)

**Erityissuorittajat:**
6. **request_info_executor** - 🆕 `RequestInfoExecutor`, joka keskeyttää työnkulun odottamaan ihmisen syötettä
7. **decision_manager** - 🆕 Räätälöity suorittaja, joka ohjaa vastauksen perusteella (jo määritelty yllä)


In [ ]:
# Agent 1: Check availability with tool (same as conditional workflow)
availability_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: NEW - Prepare human confirmation request
confirmation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user's requested destination has no available hotel rooms. "
            "Create a polite message asking if they would like to see alternative destinations nearby. "
            "Return a JSON with: destination (the unavailable city), and question (a friendly yes/no question). "
            "Keep the question concise and friendly."
        ),
        default_options={"response_format": ConfirmationQuestion},  # Structured JSON output
    ),
    id="confirmation_agent",
)

# Agent 3: Suggest alternative (when user says yes)
alternative_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 4: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

# Agent 5: NEW - Handle cancellation when user declines alternatives
class CancellationMessage(BaseModel):
    """Message when user declines alternatives."""
    status: str
    message: str

cancellation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user has declined to see alternative hotel destinations. "
            "Create a polite cancellation message. "
            "Return JSON with: status (should be 'cancelled'), and message (a friendly acknowledgment). "
            "Keep the message brief and understanding."
        ),
        default_options={"response_format": CancellationMessage},
    ),
    id="cancellation_agent",
)

# DecisionManager instance - pauses for human input (request_info) and routes on the reply
decision_manager = DecisionManager(id="decision_manager")

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created Workflow Components:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>confirmation_agent</strong> 🆕 - Prepares human confirmation request</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
            <li><strong>cancellation_agent</strong> 🆕 - Handles user declining alternatives</li>
            <li><strong>request_info_executor</strong> 🆕 - Pauses workflow for human input</li>
            <li><strong>decision_manager</strong> 🆕 - Routes based on human response</li>
        </ul>
    </div>
""")
)


## Vaihe 9: Rakenna työnkulku ihmisen kanssa mukana

Rakennamme nyt työnkulun kaavion, jossa on **ehdollinen reititys** ja ihmisen mukanaolo-polku:

**Työnkulun rakenne:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙                    ↘
[no_availability]        [has_availability]
        ↓                        ↓
confirmation_agent          booking_agent
        ↓                        ↓
prepare_human_request      display_result
        ↓
request_info_executor (PAUSE)
        ↓
decision_manager
   ↙         ↘
[yes]        [no]
   ↓           ↓
alternative  cancellation
   ↓           ↓
display_result
```

**Keskeiset reunat:**
- `availability_agent → confirmation_agent` (kun ei ole huoneita)
- `confirmation_agent → prepare_human_request` (muutos tyyppiin)
- `prepare_human_request → request_info_executor` (tauko ihmiselle)
- `request_info_executor → decision_manager` (aina - tarjoaa RequestResponse)
- `decision_manager → alternative_agent` (kun käyttäjä sanoo "kyllä")
- `decision_manager → cancellation_agent` (kun käyttäjä sanoo "ei")
- `availability_agent → booking_agent` (kun huoneita on saatavilla)
- Kaikki polut päättyvät solmuun `display_result`


In [ ]:
# Build the workflow with human-in-the-loop routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    
    # NO AVAILABILITY PATH (with human-in-the-loop)
    .add_edge(availability_agent, confirmation_agent, condition=no_availability_condition)
    .add_edge(confirmation_agent, decision_manager)  # decision_manager pauses via ctx.request_info
    
    # Decision manager routes based on user response
    .add_edge(decision_manager, alternative_agent, condition=user_wants_alternatives_condition)
    .add_edge(decision_manager, cancellation_agent, condition=user_declines_alternatives_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(cancellation_agent, display_result)
    
    # HAS AVAILABILITY PATH (no human input needed)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Human-in-the-Loop Routing:</strong><br>
            • If <strong>NO availability</strong> → confirmation_agent → prepare_human_request → request_info_executor → <strong>PAUSE FOR HUMAN</strong> → decision_manager<br>
            &nbsp;&nbsp;• If user says <strong>YES</strong> → alternative_agent → display_result<br>
            &nbsp;&nbsp;• If user says <strong>NO</strong> → cancellation_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result (no human input needed)
        </p>
    </div>
""")
)

## Vaihe 10: Suorita testitapaus 1 - Kaupunki ILMAN saatavuutta (Pariisissa ihmisen vahvistuksella)

Tämä testi demonstroi **täydellistä ihmisen osallistavan silmukan sykliä**:

1. Pyydä hotelli Pariisista
2. availability_agent tarkistaa → Ei huoneita
3. confirmation_agent luo ihmiskäyttäjälle suunnatun kysymyksen
4. request_info_executor **keskeyttää työnkulun** ja lähettää `RequestInfoEvent`-tapahtuman
5. **Sovellus havaitsee tapahtuman ja kehottaa käyttäjää konsolissa**
6. Käyttäjä kirjoittaa "kyllä" tai "ei"
7. Sovellus lähettää vastauksen `send_responses_streaming()`-funktion kautta
8. decision_manager reitittää vastauksen perusteella
9. Lopullinen tulos näytetään

**Keskeinen malli:**
- Käytä `workflow.run_stream()` ensimmäisellä iteraatiolla
- Käytä `workflow.send_responses_streaming(pending_responses)` myöhemmillä iteraatioilla
- Kuuntele `RequestInfoEvent`:ia tunnistaaksesi, milloin tarvitaan ihmisen syötettä
- Kuuntele `WorkflowOutputEvent`:ia saadaksesi lopulliset tulokset


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability - Human-in-the-Loop)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → confirmation_agent → request_info_executor → <strong>PAUSE</strong> → decision_manager → (depends on user input)</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], 
    should_respond=True
)

# Human-in-the-loop execution pattern.
# We script the human's answer ("yes") instead of input() so the notebook runs unattended.
# In a real app, replace SCRIPTED_ANSWER with input() or a UI callback.
SCRIPTED_ANSWER = "yes"
workflow_output: str | None = None

print("\n🔄 Starting human-in-the-loop workflow...")
print("=" * 60)

# First run streams events; resume with run(stream=True, responses=...) after each pause.
stream = workflow.run(request_paris, stream=True)
while True:
    requests: list[tuple[str, HumanFeedbackRequest]] = []
    async for event in stream:
        if event.type == "request_info" and isinstance(event.data, HumanFeedbackRequest):
            print(f"\n⏸️  WORKFLOW PAUSED - Human input requested!")
            print(f"   Request ID: {event.request_id}")
            print(f"   Destination: {event.data.destination}")
            print(f"   Question: {event.data.prompt}")
            requests.append((event.request_id, event.data))
        elif event.type == "output":
            workflow_output = str(event.data)
            print(f"\n✅ Workflow completed with output!")

    if not requests:
        break

    # Provide the (scripted) human answer for each pending request.
    responses: dict[str, str] = {}
    for req_id, req in requests:
        answer = SCRIPTED_ANSWER
        print(f"\n📝 (scripted) You answered: {answer}")
        responses[req_id] = answer

    stream = workflow.run(stream=True, responses=responses)

print(f"\n{'='*60}")
print(f"🏆 FINAL WORKFLOW OUTPUT:")
print(f"{'='*60}")

# Display final result
if workflow_output:
    # Try to parse as JSON for pretty display
    try:
        result_data = json.loads(workflow_output)
        if "alternative_destination" in result_data:
            result_obj = AlternativeResult.model_validate_json(workflow_output)
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> ✅ Accepted alternatives</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_obj.alternative_destination}</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_obj.reason}</p>
                    </div>
                </div>
            """)
            )
        else:
            # User declined
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #f44336 0%, #e91e63 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(244,67,54,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> 🚫 Declined alternatives</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Result:</strong> Booking request cancelled</p>
                    </div>
                </div>
            """)
            )
    except:
        print(workflow_output)


🔄 Starting human-in-the-loop workflow...

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: 032c8fce-b9d1-400e-ba8d-afd2248e2926
   Destination: Paris

💬 QUESTION FOR YOU:
   Unfortunately, there are no rooms available in Paris. Would you like to explore nearby alternative destinations?

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: cf48dad0-ee5e-4f60-8806-341a7a292bd4
   Destination: Paris

💬 QUESTION FOR YOU:
   I'm sorry to inform you that there are no available hotel rooms in Paris. Would you like me to suggest nearby alternative destinations?

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'


## Vaihe 11: Suorita testitapaus 2 - Kaupunki SAATAVUUDELLA (Tukholma - Ei ihmisosallistumista tarvittu)

Tämä testi havainnollistaa **suoraa polkua**, kun huoneita on saatavilla:

1. Pyydä hotelli Tukholmasta
2. availability_agent tarkastaa → Huoneita saatavilla ✅
3. booking_agent ehdottaa varausta
4. display_result näyttää vahvistuksen
5. **Ei ihmisen syötettä vaadita!**

Työnkulku ohittaa ihmisen osallistumisen polun kokonaan, kun huoneita on saatavilla.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability - No Human Input)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result (direct, no pause)</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], 
    should_respond=True
)

# Run the workflow (should complete without human input)
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm - No Human Input)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0 0 10px 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <p style='margin: 10px 0 0 0; font-size: 12px; color: #999; font-style: italic;'>Note: No human input was requested because rooms were available!</p>
            </div>
        </div>
    """)
    )

## Keskeiset opit ja ihminen silmukassa -parhaat käytännöt

### ✅ Mitä opit:

#### 1. **RequestInfoExecutor-malli**
Microsoft Agent Frameworkin ihminen silmukassa -malli käyttää kolmea keskeistä osaa:
- `RequestInfoExecutor` - Keskeyttää työnkulun ja lähettää tapahtumia
- `RequestInfoMessage` - Tyypitettyjen pyyntötietojen perusluokka (johdannaista tästä!)
- `RequestResponse` - Yhdistää ihmisen vastaukset alkuperäisiin pyyntöihin

**Tärkeä ymmärrys:**
- `RequestInfoExecutor` ei kerää syötettä itse – se vain keskeyttää työnkulun
- Sovelluskoodisi täytyy kuunnella `RequestInfoEvent`-tapahtumia ja kerätä syöte
- Sinun täytyy kutsua `send_responses_streaming()` sanakirjalla, joka yhdistää `request_id`:n käyttäjän vastaukseen

#### 2. **Streaming-suorituksen malli**
```python
# Ensimmäinen iteraatio
stream = workflow.run_stream(initial_request)

# Seuraavat iteraatiot (ihmisen syötteen jälkeen)
stream = workflow.send_responses_streaming(pending_responses)

# Käsittele tapahtumat aina
events = [event async for event in stream]
```

#### 3. **Tapahtumaohjattu arkkitehtuuri**
Kuuntele tiettyjä tapahtumia työnkulun hallintaan:
- `RequestInfoEvent` - Ihmisen syötettä tarvitaan (työnkulku keskeytetty)
- `WorkflowOutputEvent` - Lopullinen tulos saatavilla (työnkulku valmis)
- `WorkflowStatusEvent` - Tilamuutokset (KESKEN, ODOTTAA PYÖRIMISTÄ, jne.)

#### 4. **Mukautetut suorittajat @handlerilla**
`DecisionManager` näyttää, miten luoda suorittajia, jotka:
- Käyttävät `@handler`-koristetta paljastaakseen menetelmiä työnkulun vaiheina
- Ottavat vastaan tyypitettyjä viestejä (esim. `RequestResponse[HumanFeedbackRequest, str]`)
- Reitittävät työnkulun lähettämällä viestejä muille suorittajille
- Pääsevät käsiksi kontekstiin `WorkflowContext`:in kautta

#### 5. **Ehtoinen reititys ihmispäätöksillä**
Voit luoda ehtofunktioita, jotka arvioivat ihmisen vastauksia:
```python
def user_wants_alternatives_condition(message: Any) -> bool:
    response_text = message.agent_run_response.text.lower()
    return response_text == "yes"
```

### 🎯 Käytännön sovellukset:

1. **Hyväksyntätyönkulut**
   - Saat esimiehen hyväksynnän ennen kuluraporttien käsittelyä
   - Vaatimuksena ihmisen tarkastus ennen automaattisten sähköpostien lähettämistä
   - Vahvista arvokkaat tapahtumat ennen toteutusta

2. **Sisällön valvonta**
   - Merkitse kyseenalaiset sisällöt ihmisen tarkastettavaksi
   - Pyydä moderaattoreita tekemään lopulliset päätökset harmaissa alueissa
   - Siirrä tapauksia ihmisille, kun tekoälyn varmuus on matala

3. **Asiakaspalvelu**
   - Anna tekoälyn hoitaa rutiinikysymykset automaattisesti
   - Ohjaa monimutkaiset tapaukset ihmisasiakaspalvelijoille
   - Kysy asiakkaalta, haluaako hän puhua ihmisen kanssa

4. **Datan käsittely**
   - Pyydä ihmisiä ratkaisemaan epäselvät tiedot
   - Vahvista tekoälyn tulkinnat epäselvistä asiakirjoista
   - Anna käyttäjille mahdollisuus valita useiden kelvollisten tulkintojen välillä

5. **Turvallisuuskriittiset järjestelmät**
   - Vaadi ihmisen vahvistus ennen peruuttamattomia toimia
   - Hanki hyväksyntä ennen arkaluonteisiin tietoihin pääsyä
   - Vahvista päätökset säännellyillä aloilla (terveydenhuolto, finanssi)

6. **Interaktiiviset agentit**
   - Rakenna keskustelubotteja, jotka esittävät jatkokysymyksiä
   - Luo ohjattuja prosesseja käyttäjille monimutkaisissa tehtävissä
   - Suunnittele agentteja, jotka tekevät yhteistyötä ihmisten kanssa vaihe vaiheelta

### 🔄 Vertailu: Ihminen silmukassa vs ilman

| Ominaisuus | Ehtoihin perustuva työnkulku | Ihminen silmukassa -työnkulku |
|---------|---------------------|---------------------------|
| **Suoritus** | Yksittäinen `workflow.run()` | Silmukka `run_stream()` + `send_responses_streaming()` |
| **Käyttäjän syöte** | Ei syötettä (täysin automatisoitu) | Vuorovaikutteiset kehotteet `input()` tai käyttöliittymän kautta |
| **Komponentit** | Agentit + suorittajat | + RequestInfoExecutor + DecisionManager |
| **Tapahtumat** | Vain AgentExecutorResponse | RequestInfoEvent, WorkflowOutputEvent, jne. |
| **Keskeytykset** | Ei keskeytyksiä | Työnkulku keskeytyy RequestInfoExecutorin kohdalla |
| **Ihmisen kontrolli** | Ei ihmisen kontrollia | Ihmiset tekevät avainpäätökset |
| **Käyttötapaus** | Automaatio | Yhteistyö ja valvonta |

### 🚀 Edistyneet mallit:

#### Useita ihmispäätöspisteitä
Voit käyttää useita `RequestInfoExecutor`-solmuja samassa työnkulussa:
```python
.add_edge(agent1, request_info_1)  # Ensimmäinen ihmisen päätös
.add_edge(decision_manager_1, agent2)
.add_edge(agent2, request_info_2)  # Toinen ihmisen päätös
.add_edge(decision_manager_2, final_agent)
```

#### Aikarajoitusten käsittely
Toteuta aikakatkaisut ihmisen vastauksiin:
```python
import asyncio

try:
    answer = await asyncio.wait_for(
        asyncio.to_thread(input, "Enter yes/no: "),
        timeout=60.0
    )
except asyncio.TimeoutError:
    answer = "no"  # Oletuksena turvallinen vaihtoehto
```

#### Kehittynyt käyttöliittymäintegraatio
`input()` sijaan integroi web-käyttöliittymään, Slackiin, Teamsiin jne.:
```python
if isinstance(event, RequestInfoEvent):
    # Lähetä ilmoitus käyttäjän mieluisalle kanavalle
    await slack_client.send_message(
        user_id=current_user,
        text=event.data.prompt,
        request_id=event.request_id
    )
```

#### Ehtoinen ihminen silmukassa
Kysy ihmisen syötettä vain tietyissä tilanteissa:
```python
def needs_human_approval_condition(message: Any) -> bool:
    # Reititä ihmiseen vain, jos luottamus on matala tai arvo on korkea
    if result.confidence < 0.7 or result.value > 10000:
        return True
    return False
```

### ⚠️ Parhaat käytännöt:

1. **Johdannaista aina RequestInfoMessagesta**
   - Tarjoaa tyyppiturvallisuuden ja validoinnin
   - Mahdollistaa rikkaan kontekstin käyttöliittymän näyttöön
   - Selkeyttää kunkin pyyntötason tarkoituksen

2. **Käytä kuvaavia kehotteita**
   - Sisällytä konteksti siitä, mitä kysyt
   - Selitä kunkin valinnan seuraukset
   - Pidä kysymykset yksinkertaisina ja selkeinä

3. **Käsittele odottamattomat syötteet**
   - Validoi käyttäjän vastaukset
   - Tarjoa oletusarvot virheellisille syötteille
   - Anna selkeät virheilmoitukset

4. **Seuraa pyyntöjen tunnisteita**
   - Hyödynnä korrelaatiota request_id:n ja vastausten välillä
   - Älä yritä hallita tilaa manuaalisesti

5. **Suunnittele estämättömäksi**
   - Älä estä säikeitä odottamalla syötettä
   - Käytä asynkronisia malleja läpi koko sovelluksen
   - Tue samanaikaisia työnkulun instansseja

### 📚 Liittyvät käsitteet:

- **Agentin välimuisti** - Agenttikutsujen sieppaaminen (edellinen muistio)
- **Työnkulun tilanhallinta** - Työnkulun tilan säilyttäminen suoritusten välillä
- **Moni-agentti yhteistyö** - Ihminen silmukassa yhdistettynä agenttitiimeihin
- **Tapahtumaohjatut arkkitehtuurit** - Reaktiivisten järjestelmien rakentaminen tapahtumien avulla

---

### 🎓 Onnittelut!

Olet hallinnut ihmisen silmukassa -työnkulut Microsoft Agent Frameworkissa! Osaat nyt:
- ✅ Keskeyttää työnkulut ihmisen syötteen keräämiseksi
- ✅ Käyttää RequestInfoExecutoria ja RequestInfoMessagea
- ✅ Käsitellä streaming-suoritusta tapahtumien avulla
- ✅ Luoda mukautettuja suorittajia @handlerilla
- ✅ Reitittää työnkulkuja ihmisten päätösten mukaan
- ✅ Rakentaa vuorovaikutteisia tekoälyagentteja, jotka tekevät yhteistyötä ihmisten kanssa

**Tämä on kriittinen malli luotettavien, hallittavien tekoälyjärjestelmien rakentamisessa!** 🚀


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, otathan huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
